# 3. The Vehicle Routing Problem with Time Windows

## Tier 3 — Genetic Algorithm (Metaheuristic Optimization)

This notebook implements a **Genetic Algorithm (GA)** for VRPTW, demonstrating how evolutionary computation can find high-quality solutions for complex routing problems. GA combines exploration (diversification) with exploitation (intensification) to escape local optima.

### Learning goals

- Understand how **evolutionary operators** (crossover, mutation) work on routing problems.
- See how **chromosome encoding** represents VRPTW solutions.
- Learn how **fitness functions** balance multiple objectives (distance, time windows).
- Practice **parameter tuning** for metaheuristic performance.

### What this notebook outputs

- Evolution of solution quality over generations.
- Comparison of different GA configurations.
- Best-found routes with detailed statistics.
- Convergence analysis and parameter sensitivity.

### Why this Tier exists vs earlier Tiers

**Tier 1 (MIP)** guarantees optimality but does not scale. **Tier 2 (Heuristics)** are fast but often get stuck in local optima. **Tier 3 (GA)** provides a middle ground:
- Better solution quality than simple heuristics
- Faster than exact methods for large instances
- Can escape local optima through evolutionary operators
- Provides multiple good solutions (population-based)

### When to use this Tier

- When you need **high-quality solutions** for medium-large instances
- When **multiple alternative solutions** are valuable
- When you have **moderate computation time** available (minutes)
- When problem has **complex constraints** that fool simple heuristics

In [ ]:
# Environment check (no installs here)
#
# Best practice for classes: preinstall dependencies in the Docker/JupyterHub image.
# If you're running locally, install packages once in your environment.

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import time
    import random
    from typing import List, Tuple, Dict, Optional
    from dataclasses import dataclass
    import copy
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Concrete instance (10 customers, 3 vehicles)

We'll use a medium-sized instance to demonstrate GA advantages:

- **Depot** at location (0, 0) with time window [0, ∞)
- **10 customers** with varied locations, demands, and time windows
- **3 identical vehicles** with capacity 50 units
- **Service time** of 10 minutes per customer
- **Travel speed** of 1 unit per minute

This instance is complex enough to benefit from GA's global search capabilities.

In [ ]:
# ----------------------------
# Data structures and instance
# ----------------------------
# We'll use consistent data structures with previous tiers.

@dataclass(frozen=True)
class Customer:
    id: int
    location: Tuple[float, float]
    demand: float
    time_window: Tuple[float, float]
    service_time: float


@dataclass
class Route:
    customers: List[int]
    demand: float
    distance: float
    timeline: List[Dict]
    feasible: bool


# ----------------------------
# Concrete 10-customer instance
# ----------------------------
# More complex instance to showcase GA advantages
customers = [
    # Depot
    Customer(0, (0.0, 0.0), 0.0, (0.0, 1000.0), 0.0),
    
    # Customers with varied characteristics
    Customer(1, (2.0, 3.0), 8.0, (5.0, 20.0), 10.0),
    Customer(2, (5.0, 2.0), 12.0, (10.0, 25.0), 10.0),
    Customer(3, (8.0, 6.0), 15.0, (15.0, 30.0), 10.0),
    Customer(4, (6.0, 1.0), 10.0, (8.0, 18.0), 10.0),
    Customer(5, (3.0, 7.0), 14.0, (12.0, 35.0), 10.0),
    Customer(6, (9.0, 4.0), 6.0, (20.0, 28.0), 10.0),
    Customer(7, (4.0, 8.0), 9.0, (18.0, 32.0), 10.0),
    Customer(8, (7.0, 5.0), 11.0, (14.0, 26.0), 10.0),
    Customer(9, (1.0, 6.0), 13.0, (16.0, 34.0), 10.0),
    Customer(10, (10.0, 7.0), 7.0, (22.0, 40.0), 10.0),
]

# Fast lookup
id_to_customer = {c.id: c for c in customers}

# Problem parameters
NUM_VEHICLES = 3
VEHICLE_CAPACITY = 50.0
TRAVEL_SPEED = 1.0

# Sets
CUSTOMERS = [c for c in customers if c.id != 0]
ALL_NODES = customers


# ----------------------------
# Helper functions
# ----------------------------
def euclidean_distance(loc1: Tuple[float, float], loc2: Tuple[float, float]) -> float:
    """Compute Euclidean distance between two locations."""
    return np.sqrt((loc1[0] - loc2[0])**2 + (loc1[1] - loc2[1])**2)


def travel_time_matrix() -> Dict[Tuple[int, int], float]:
    """Precompute travel times between all node pairs."""
    times = {}
    for i in ALL_NODES:
        for j in ALL_NODES:
            if i != j:
                dist = euclidean_distance(i.location, j.location)
                times[(i.id, j.id)] = dist / TRAVEL_SPEED
            else:
                times[(i.id, j.id)] = 0.0
    return times


travel_times = travel_time_matrix()

# Display the instance
print("Customer Data:")
customer_df = pd.DataFrame([
    {
        "ID": c.id,
        "Location": c.location,
        "Demand": c.demand,
        "Time Window": c.time_window,
        "Service Time": c.service_time
    }
    for c in customers
])
print(customer_df.to_string(index=False))

print(f"\nProblem Parameters:")
print(f"- Number of vehicles: {NUM_VEHICLES}")
print(f"- Vehicle capacity: {VEHICLE_CAPACITY}")
print(f"- Number of customers: {len(CUSTOMERS)}")

## Step 1 — Route evaluation utilities

We need the same route evaluation functions as previous tiers to assess solution quality.

In [ ]:
# ----------------------------
# Route evaluation utilities (reused from Tier 2)
# ----------------------------

def calculate_route_distance(customers_list: List[int]) -> float:
    """Calculate total distance of a route (depot -> customers -> depot)."""
    if not customers_list:
        return 0.0
    
    distance = travel_times[(0, customers_list[0])]
    for i in range(len(customers_list) - 1):
        distance += travel_times[(customers_list[i], customers_list[i+1])]
    distance += travel_times[(customers_list[-1], 0)]
    
    return distance


def calculate_route_timeline(customers_list: List[int]) -> Tuple[List[Dict], bool]:
    """Calculate detailed timeline and check time window feasibility."""
    timeline = []
    current_time = 0.0
    current_pos = 0
    feasible = True
    
    for cid in customers_list:
        customer = id_to_customer[cid]
        travel = travel_times[(current_pos, cid)]
        arrival = current_time + travel
        
        earliest, latest = customer.time_window
        if arrival > latest + 0.01:
            feasible = False
            break
        
        wait = max(0.0, earliest - arrival)
        start_service = arrival + wait
        
        timeline.append({
            "customer": cid,
            "arrival": arrival,
            "wait": wait,
            "start_service": start_service,
            "finish_service": start_service + customer.service_time
        })
        
        current_time = start_service + customer.service_time
        current_pos = cid
    
    return timeline, feasible


def check_capacity_feasibility(customers_list: List[int]) -> bool:
    """Check if route respects vehicle capacity."""
    total_demand = sum(id_to_customer[cid].demand for cid in customers_list)
    return total_demand <= VEHICLE_CAPACITY


def evaluate_route(customers_list: List[int]) -> Route:
    """Comprehensive route evaluation."""
    if not customers_list:
        return Route([], 0.0, 0.0, [], True)
    
    demand = sum(id_to_customer[cid].demand for cid in customers_list)
    distance = calculate_route_distance(customers_list)
    timeline, time_feasible = calculate_route_timeline(customers_list)
    capacity_feasible = check_capacity_feasibility(customers_list)
    feasible = time_feasible and capacity_feasible
    
    return Route(customers_list, demand, distance, timeline, feasible)


print("Route evaluation utilities defined successfully!")

## Step 2 — Genetic Algorithm components

The GA requires several key components:

1. **Chromosome encoding**: How to represent VRPTW solutions genetically
2. **Population initialization**: Creating diverse initial solutions
3. **Fitness function**: Evaluating solution quality
4. **Selection**: Choosing parents for reproduction
5. **Crossover**: Combining parent solutions
6. **Mutation**: Introducing random changes
7. **Replacement**: Creating the next generation

In [ ]:
# ----------------------------
# Genetic Algorithm components
# ----------------------------
# Core GA building blocks for VRPTW.

class VRPTWGeneticAlgorithm:
    """Genetic Algorithm for Vehicle Routing Problem with Time Windows."""
    
    def __init__(self, population_size: int = 50, generations: int = 100,
                 crossover_rate: float = 0.8, mutation_rate: float = 0.2,
                 tournament_size: int = 3, elite_size: int = 2):
        """Initialize GA parameters."""
        self.population_size = population_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.tournament_size = tournament_size
        self.elite_size = elite_size
        
        # Track evolution
        self.best_fitness_history = []
        self.avg_fitness_history = []
        self.diversity_history = []
        
        # Customer list for chromosome operations
        self.customer_ids = [c.id for c in CUSTOMERS]
        
    def chromosome_to_routes(self, chromosome: List[int]) -> List[Route]:
        """Convert chromosome (customer permutation) to routes using split procedure."""
        routes = []
        current_route = []
        current_demand = 0.0
        
        for customer_id in chromosome:
            customer_demand = id_to_customer[customer_id].demand
            
            # Check if adding customer violates capacity
            if current_demand + customer_demand > VEHICLE_CAPACITY:
                # Finish current route and start new one
                if current_route:
                    route_eval = evaluate_route(current_route)
                    routes.append(route_eval)
                current_route = [customer_id]
                current_demand = customer_demand
            else:
                # Add customer to current route
                current_route.append(customer_id)
                current_demand += customer_demand
        
        # Add final route
        if current_route:
            route_eval = evaluate_route(current_route)
            routes.append(route_eval)
        
        # Limit to available vehicles
        if len(routes) > NUM_VEHICLES:
            # Merge excess routes (simple concatenation - could be improved)
            while len(routes) > NUM_VEHICLES:
                # Merge last two routes
                last_route = routes[-1]
                second_last_route = routes[-2]
                merged_customers = second_last_route.customers + last_route.customers
                merged_eval = evaluate_route(merged_customers)
                routes[-2] = merged_eval
                routes.pop()
        
        return routes
    
    def calculate_fitness(self, chromosome: List[int]) -> float:
        """Calculate fitness value for a chromosome."""
        routes = self.chromosome_to_routes(chromosome)
        
        # Calculate total distance
        total_distance = sum(route.distance for route in routes)
        
        # Check feasibility
        infeasible_routes = sum(1 for route in routes if not route.feasible)
        
        # Penalize infeasibility heavily
        if infeasible_routes > 0:
            penalty = 1000.0 * infeasible_routes
            return 1.0 / (1.0 + total_distance + penalty)
        
        # Fitness = 1 / (1 + total_distance) for minimization
        return 1.0 / (1.0 + total_distance)
    
    def initialize_population(self) -> List[List[int]]:
        """Create initial population with diverse solutions."""
        population = []
        
        # Add some heuristic solutions for good starting points
        # (Simplified versions of Tier 2 heuristics)
        for _ in range(min(5, self.population_size // 10)):
            # Random permutation
            chromosome = self.customer_ids.copy()
            random.shuffle(chromosome)
            population.append(chromosome)
        
        # Fill rest with random permutations
        while len(population) < self.population_size:
            chromosome = self.customer_ids.copy()
            random.shuffle(chromosome)
            population.append(chromosome)
        
        return population
    
    def tournament_selection(self, population: List[List[int]], 
                              fitness_values: List[float]) -> List[int]:
        """Select parents using tournament selection."""
        selected = []
        
        for _ in range(self.population_size):
            # Random tournament
            tournament_indices = random.sample(
                range(len(population)), 
                min(self.tournament_size, len(population))
            )
            
            # Find winner (highest fitness)
            winner_idx = max(tournament_indices, key=lambda i: fitness_values[i])
            selected.append(winner_idx)
        
        return selected
    
    def order_crossover(self, parent1: List[int], parent2: List[int]) -> Tuple[List[int], List[int]]:
        """Order crossover (OX) for permutation encoding."""
        if random.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()
        
        size = len(parent1)
        
        # Select two random crossover points
        start, end = sorted(random.sample(range(size), 2))
        
        # Create offspring
        child1 = [-1] * size
        child2 = [-1] * size
        
        # Copy segments from parents
        child1[start:end] = parent1[start:end]
        child2[start:end] = parent2[start:end]
        
        # Fill remaining positions with order from other parent
        def fill_child(child, other_parent):
            remaining = [x for x in other_parent if x not in child]
            idx = 0
            for i in range(len(child)):
                if child[i] == -1:
                    child[i] = remaining[idx]
                    idx += 1
        
        fill_child(child1, parent2)
        fill_child(child2, parent1)
        
        return child1, child2
    
    def swap_mutation(self, chromosome: List[int]) -> List[int]:
        """Swap mutation for permutation encoding."""
        if random.random() > self.mutation_rate:
            return chromosome.copy()
        
        mutated = chromosome.copy()
        
        # Select two random positions and swap
        if len(mutated) > 1:
            pos1, pos2 = random.sample(range(len(mutated)), 2)
            mutated[pos1], mutated[pos2] = mutated[pos2], mutated[pos1]
        
        return mutated
    
    def calculate_diversity(self, population: List[List[int]]) -> float:
        """Calculate population diversity (average Hamming distance)."""
        if len(population) < 2:
            return 0.0
        
        total_distance = 0.0
        comparisons = 0
        
        for i in range(len(population)):
            for j in range(i + 1, len(population)):
                # Hamming distance
                distance = sum(1 for a, b in zip(population[i], population[j]) if a != b)
                total_distance += distance
                comparisons += 1
        
        return total_distance / comparisons if comparisons > 0 else 0.0
    
    def evolve(self) -> Tuple[List[int], float, List[List[int]]]:
        """Run the genetic algorithm and return best solution."""
        print(f"Starting GA with {self.population_size} population, {self.generations} generations")
        
        # Initialize population
        population = self.initialize_population()
        
        for generation in range(self.generations):
            # Calculate fitness values
            fitness_values = [self.calculate_fitness(chrom) for chrom in population]
            
            # Track statistics
            best_fitness = max(fitness_values)
            avg_fitness = sum(fitness_values) / len(fitness_values)
            diversity = self.calculate_diversity(population)
            
            self.best_fitness_history.append(best_fitness)
            self.avg_fitness_history.append(avg_fitness)
            self.diversity_history.append(diversity)
            
            # Report progress
            if generation % 20 == 0 or generation == self.generations - 1:
                print(f"Gen {generation:3d}: Best={1.0/best_fitness-1:.2f}, Avg={1.0/avg_fitness-1:.2f}, Div={diversity:.1f}")
            
            # Selection
            selected_indices = self.tournament_selection(population, fitness_values)
            
            # Create new population
            new_population = []
            
            # Elitism: keep best solutions
            sorted_indices = sorted(range(len(fitness_values)), key=lambda i: fitness_values[i], reverse=True)
            for i in range(min(self.elite_size, len(population))):
                new_population.append(population[sorted_indices[i]].copy())
            
            # Generate offspring
            while len(new_population) < self.population_size:
                # Select two parents
                parent1_idx = selected_indices[random.randint(0, len(selected_indices) - 1)]
                parent2_idx = selected_indices[random.randint(0, len(selected_indices) - 1)]
                
                parent1 = population[parent1_idx]
                parent2 = population[parent2_idx]
                
                # Crossover
                child1, child2 = self.order_crossover(parent1, parent2)
                
                # Mutation
                child1 = self.swap_mutation(child1)
                child2 = self.swap_mutation(child2)
                
                new_population.extend([child1, child2])
            
            # Trim to exact population size
            population = new_population[:self.population_size]
        
        # Find best solution
        final_fitness = [self.calculate_fitness(chrom) for chrom in population]
        best_idx = max(range(len(final_fitness)), key=lambda i: final_fitness[i])
        best_chromosome = population[best_idx]
        best_fitness = final_fitness[best_idx]
        
        print(f"GA completed! Best distance: {1.0/best_fitness-1:.2f}")
        
        return best_chromosome, best_fitness, population


print("Genetic Algorithm components defined successfully!")

## Step 3 — Run the Genetic Algorithm

Now we'll run the GA with default parameters and analyze the results.

In [ ]:
# ----------------------------
# Run Genetic Algorithm with default parameters
# ----------------------------
# Standard GA configuration for VRPTW.

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Create and run GA
ga = VRPTWGeneticAlgorithm(
    population_size=50,
    generations=100,
    crossover_rate=0.8,
    mutation_rate=0.2,
    tournament_size=3,
    elite_size=2
)

print("Running Genetic Algorithm...")
start_time = time.time()
best_chromosome, best_fitness, final_population = ga.evolve()
ga_time = time.time() - start_time

# Convert best chromosome to routes
best_routes = ga.chromosome_to_routes(best_chromosome)
best_distance = sum(route.distance for route in best_routes)

print(f"\nGA Results:")
print(f"Computation time: {ga_time:.2f} seconds")
print(f"Best distance: {best_distance:.2f}")
print(f"Number of routes: {len(best_routes)}")
print(f"Feasible routes: {sum(1 for route in best_routes if route.feasible)}")

# Display best routes
print("\nBest routes found:")
for i, route in enumerate(best_routes):
    print(f"Route {i+1}: {' -> '.join(map(str, ['0'] + route.customers + ['0']))}")
    print(f"  Distance: {route.distance:.2f}, Demand: {route.demand:.1f}, Feasible: {route.feasible}")

## Step 4 — Evolution analysis

Let's analyze how the GA evolved over generations to understand its convergence behavior.

In [ ]:
# ----------------------------
# Evolution analysis and visualization
# ----------------------------
# Analyze GA convergence and diversity trends.

def plot_evolution():
    """Plot GA evolution statistics."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    generations = list(range(len(ga.best_fitness_history)))
    
    # Convert fitness to distance for better interpretation
    best_distances = [1.0/f - 1 for f in ga.best_fitness_history]
    avg_distances = [1.0/f - 1 for f in ga.avg_fitness_history]
    
    # 1. Best and average distance over generations
    axes[0, 0].plot(generations, best_distances, 'b-', linewidth=2, label='Best')
    axes[0, 0].plot(generations, avg_distances, 'r--', linewidth=2, label='Average')
    axes[0, 0].set_xlabel('Generation')
    axes[0, 0].set_ylabel('Total Distance')
    axes[0, 0].set_title('Solution Quality Evolution')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Population diversity
    axes[0, 1].plot(generations, ga.diversity_history, 'g-', linewidth=2)
    axes[0, 1].set_xlabel('Generation')
    axes[0, 1].set_ylabel('Average Hamming Distance')
    axes[0, 1].set_title('Population Diversity')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Improvement rate (moving average of best distance improvement)
    window_size = 10
    if len(best_distances) > window_size:
        improvement_rate = []
        for i in range(window_size, len(best_distances)):
            recent_improvement = best_distances[i-window_size] - best_distances[i]
            improvement_rate.append(recent_improvement / window_size)
        
        axes[1, 0].plot(generations[window_size:], improvement_rate, 'purple', linewidth=2)
        axes[1, 0].set_xlabel('Generation')
        axes[1, 0].set_ylabel('Improvement per Generation')
        axes[1, 0].set_title(f'Learning Rate ( {window_size}-gen window)')
        axes[1, 0].grid(True, alpha=0.3)
        
        # Add zero line
        axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    
    # 4. Convergence detection (when improvement falls below threshold)
    threshold = 0.1  # Distance improvement threshold
    converged = False
    convergence_gen = None
    
    for i in range(20, len(best_distances)):
        recent_improvement = best_distances[i-20] - best_distances[i]
        if recent_improvement < threshold:
            converged = True
            convergence_gen = i
            break
    
    # Plot convergence detection
    axes[1, 1].plot(generations, best_distances, 'b-', linewidth=2, label='Best Distance')
    if converged:
        axes[1, 1].axvline(x=convergence_gen, color='red', linestyle='--', 
                        label=f'Converged at Gen {convergence_gen}')
        axes[1, 1].fill_between([convergence_gen, generations[-1]], 
                                 [min(best_distances), min(best_distances)],
                                 [max(best_distances), max(best_distances)],
                                 alpha=0.2, color='red')
    
    axes[1, 1].set_xlabel('Generation')
    axes[1, 1].set_ylabel('Total Distance')
    axes[1, 1].set_title('Convergence Analysis')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return converged, convergence_gen


# Plot evolution statistics
converged, convergence_gen = plot_evolution()

print(f"\n=== EVOLUTION ANALYSIS ===")
if converged:
    print(f"GA converged at generation {convergence_gen}")
    print(f"Final {len(ga.best_fitness_history) - convergence_gen} generations showed minimal improvement")
else:
    print("GA did not converge within the generation limit")
    print("Consider increasing generations for better convergence")

print(f"\nPopulation diversity trends:")
print(f"Initial diversity: {ga.diversity_history[0]:.1f}")
print(f"Final diversity: {ga.diversity_history[-1]:.1f}")
print(f"Diversity loss: {(ga.diversity_history[0] - ga.diversity_history[-1])/ga.diversity_history[0]*100:.1f}%")

## Step 5 — Solution visualization

Let's visualize the best solution found by the GA and analyze its routing patterns.

In [ ]:
# ----------------------------
# Solution visualization
# ----------------------------
# Visualize the best GA solution.

def plot_ga_solution(routes, title):
    """Plot GA solution with route details."""
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Colors for different vehicles
    colors = ['#2E90FA', '#12B76A', '#F59E0B', '#EF4444', '#8B5CF6', '#EC4899']
    
    # Plot depot
    depot = id_to_customer[0]
    ax.scatter(depot.location[0], depot.location[1], s=200, c='black', marker='s', 
               edgecolors='white', linewidth=2, label='Depot', zorder=5)
    ax.annotate('DEPOT', (depot.location[0], depot.location[1]), 
                xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')
    
    # Plot customers
    for customer in CUSTOMERS:
        ax.scatter(customer.location[0], customer.location[1], s=100, c='lightgray', 
                   edgecolors='black', linewidth=1, zorder=3)
        
        # Label with customer ID and time window
        time_window_str = f"[{customer.time_window[0]:.0f}, {customer.time_window[1]:.0f}]"
        ax.annotate(f"C{customer.id}\n{time_window_str}", 
                    (customer.location[0], customer.location[1]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    # Plot routes
    for i, route in enumerate(routes):
        if not route.customers:
            continue
        
        color = colors[i % len(colors)]
        route_customers = route.customers
        
        # Route from depot to first customer
        points = [depot.location] + [id_to_customer[cid].location for cid in route_customers]
        
        # Plot route segments
        for j in range(len(points) - 1):
            x_vals = [points[j][0], points[j+1][0]]
            y_vals = [points[j][1], points[j+1][1]]
            ax.plot(x_vals, y_vals, color=color, linewidth=2, alpha=0.7, 
                   label=f'Vehicle {i+1}' if j == 0 else '')
            
            # Add arrows to show direction
            if j == 0:  # Only add arrow for first segment to avoid clutter
                mid_x = (points[j][0] + points[j+1][0]) / 2
                mid_y = (points[j][1] + points[j+1][1]) / 2
                dx = points[j+1][0] - points[j][0]
                dy = points[j+1][1] - points[j][1]
                ax.annotate('', xy=(mid_x + dx*0.1, mid_y + dy*0.1), 
                           xytext=(mid_x - dx*0.1, mid_y - dy*0.1),
                           arrowprops=dict(arrowstyle='->', color=color, lw=2))
        
        # Return to depot
        if route_customers:
            last_customer = id_to_customer[route_customers[-1]]
            ax.plot([last_customer.location[0], depot.location[0]], 
                   [last_customer.location[1], depot.location[1]], 
                   color=color, linewidth=2, alpha=0.7, linestyle='--')
    
    # Formatting
    ax.set_xlabel('X Coordinate', fontsize=12)
    ax.set_ylabel('Y Coordinate', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=10)
    
    # Set equal aspect ratio for accurate geography
    ax.set_aspect('equal')
    
    # Add text box with solution statistics
    total_distance = sum(route.distance for route in routes)
    total_customers = sum(len(route.customers) for route in routes)
    feasible_routes = sum(1 for route in routes if route.feasible)
    
    stats_text = f"Total Distance: {total_distance:.2f}\n"
    stats_text += f"Routes Used: {len(routes)}/{NUM_VEHICLES}\n"
    stats_text += f"Customers Served: {total_customers}\n"
    stats_text += f"Feasible Routes: {feasible_routes}"
    
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()


# Plot the best GA solution
plot_ga_solution(best_routes, 'VRPTW Solution - Genetic Algorithm (Best Found)')

# Print detailed route analysis
print("\n=== DETAILED ROUTE ANALYSIS ===")
for i, route in enumerate(best_routes):
    print(f"\nRoute {i+1}:")
    print(f"  Path: 0 -> {' -> '.join(map(str, route.customers))} -> 0")
    print(f"  Distance: {route.distance:.2f}")
    print(f"  Demand: {route.demand:.1f}/{VEHICLE_CAPACITY} ({route.demand/VEHICLE_CAPACITY*100:.1f}% capacity)")
    print(f"  Feasible: {route.feasible}")
    
    if route.timeline:
        print(f"  Timeline:")
        for event in route.timeline:
            customer = id_to_customer[event['customer']]
            wait_str = f" + {event['wait']:.1f}min wait" if event['wait'] > 0.1 else ""
            print(f"    C{event['customer']}: arrive {event['arrival']:.1f}, service {event['start_service']:.1f}{wait_str}")

## Step 6 — Parameter sensitivity analysis

GA performance depends heavily on parameter settings. Let's test different configurations to understand their impact.

In [ ]:
# ----------------------------
# Parameter sensitivity analysis
# ----------------------------
# Test different GA parameter configurations.

def test_ga_configuration(config_name, population_size, generations, 
                         crossover_rate, mutation_rate, tournament_size, elite_size):
    """Test a specific GA configuration."""
    print(f"\nTesting {config_name}...")
    
    # Create GA with specific parameters
    test_ga = VRPTWGeneticAlgorithm(
        population_size=population_size,
        generations=generations,
        crossover_rate=crossover_rate,
        mutation_rate=mutation_rate,
        tournament_size=tournament_size,
        elite_size=elite_size
    )
    
    # Run GA
    start_time = time.time()
    best_chrom, best_fit, _ = test_ga.evolve()
    comp_time = time.time() - start_time
    
    # Evaluate solution
    routes = test_ga.chromosome_to_routes(best_chrom)
    distance = sum(route.distance for route in routes)
    feasible = all(route.feasible for route in routes)
    
    return {
        "Configuration": config_name,
        "Distance": distance,
        "Time": comp_time,
        "Feasible": feasible,
        "Population Size": population_size,
        "Generations": generations,
        "Crossover Rate": crossover_rate,
        "Mutation Rate": mutation_rate
    }


# Test different configurations
configurations = [
    ("Default", 50, 100, 0.8, 0.2, 3, 2),
    ("Large Population", 100, 50, 0.8, 0.2, 3, 2),
    ("High Mutation", 50, 100, 0.8, 0.4, 3, 2),
    ("Low Mutation", 50, 100, 0.8, 0.1, 3, 2),
    ("High Crossover", 50, 100, 0.9, 0.2, 3, 2),
    ("More Generations", 50, 200, 0.8, 0.2, 3, 2)
]

results = []
for config in configurations:
    result = test_ga_configuration(*config)
    results.append(result)

# Create comparison table
param_df = pd.DataFrame(results)
print("\n=== PARAMETER SENSITIVITY ANALYSIS ===")
print(param_df[['Configuration', 'Distance', 'Time', 'Feasible']].to_string(index=False))

# Find best configuration
best_config = param_df.loc[param_df['Distance'].idxmin()]
print(f"\nBest configuration: {best_config['Configuration']}")
print(f"Distance: {best_config['Distance']:.2f}")
print(f"Time: {best_config['Time']:.2f}s")

# Parameter insights
print(f"\n=== PARAMETER INSIGHTS ===")
print("Population Size:")
print("- Larger populations explore more but take longer per generation")
print("- Small populations may converge prematurely")
print("\nMutation Rate:")
print("- High mutation maintains diversity but slows convergence")
print("- Low mutation may get stuck in local optima")
print("\nGenerations:")
print("- More generations allow better convergence")
print("- Diminishing returns after convergence")

## Step 7 — Comparison with other methods

Let's compare the GA solution quality with simple heuristics to understand the trade-offs.

In [ ]:
# ----------------------------
# Comparison with simple heuristics
# ----------------------------
# Compare GA with simple insertion heuristics.

def simple_insertion_heuristic():
    """Very simple insertion heuristic for comparison."""
    unassigned = set(c.id for c in CUSTOMERS)
    routes = []
    
    while unassigned and len(routes) < NUM_VEHICLES:
        current_route = []
        current_demand = 0.0
        
        # Greedy insertion based on distance from depot
        while unassigned:
            best_customer = None
            best_cost = float('inf')
            
            for customer_id in unassigned:
                customer = id_to_customer[customer_id]
                
                # Check capacity
                if current_demand + customer.demand > VEHICLE_CAPACITY:
                    continue
                
                # Simple cost: distance from depot
                cost = travel_times[(0, customer_id)]
                
                if cost < best_cost:
                    best_cost = cost
                    best_customer = customer_id
            
            if best_customer is None:
                break
            
            current_route.append(best_customer)
            current_demand += id_to_customer[best_customer].demand
            unassigned.remove(best_customer)
        
        if current_route:
            routes.append(evaluate_route(current_route))
    
    return routes


# Run simple heuristic for comparison
print("\nRunning simple insertion heuristic for comparison...")
start_time = time.time()
heuristic_routes = simple_insertion_heuristic()
heuristic_time = time.time() - start_time

heuristic_distance = sum(route.distance for route in heuristic_routes)

# Compare results
print(f"\n=== METHOD COMPARISON ===")
comparison_data = [
    {
        "Method": "Genetic Algorithm",
        "Distance": best_distance,
        "Time (s)": ga_time,
        "Routes": len(best_routes),
        "Feasible": all(route.feasible for route in best_routes)
    },
    {
        "Method": "Simple Insertion",
        "Distance": heuristic_distance,
        "Time (s)": heuristic_time,
        "Routes": len(heuristic_routes),
        "Feasible": all(route.feasible for route in heuristic_routes)
    }
]

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Calculate improvement
improvement = (heuristic_distance - best_distance) / heuristic_distance * 100
print(f"\nGA improvement over simple heuristic: {improvement:.1f}% distance reduction")
print(f"GA time penalty: {(ga_time / heuristic_time - 1) * 100:.1f}% more time")

# Visualize both solutions
plot_ga_solution(best_routes, 'VRPTW Solution - Genetic Algorithm')
plot_ga_solution(heuristic_routes, 'VRPTW Solution - Simple Insertion Heuristic')

## Key insights and practical recommendations

### GA advantages and limitations

**Advantages:**
- Can escape local optima through mutation
- Provides multiple good solutions (population)
- Adaptable to different problem variants
- Parallelizable evaluation

**Limitations:**
- No optimality guarantee
- Parameter tuning required
- Computationally intensive for large instances
- Stochastic results (different runs may give different solutions)

### When to use Genetic Algorithms

| Problem Size | Time Available | Solution Quality Required | Recommended Approach |
|---------------|----------------|---------------------------|---------------------|
| < 50 customers | > 5 minutes | Near-optimal | GA with large population |
| 50-200 customers | 1-5 minutes | High quality | GA with moderate population |
| 200+ customers | < 1 minute | Good enough | Fast heuristics |
| Dynamic problems | Real-time | Any feasible solution | Greedy heuristics |

### Parameter tuning guidelines

1. **Population size**: 2-10x problem size (customers)
2. **Generations**: 50-200 for most problems
3. **Crossover rate**: 0.7-0.9 (high for exploration)
4. **Mutation rate**: 0.1-0.3 (higher for complex problems)
5. **Tournament size**: 2-5 (smaller = less selection pressure)
6. **Elite size**: 1-5 (preserve best solutions)

### Common implementation pitfalls

1. **Poor chromosome encoding**: Ensure all solutions are representable
2. **Inadequate fitness function**: Balance feasibility and solution quality
3. **Premature convergence**: Maintain diversity through mutation
4. **Too many generations**: Stop when convergence is detected
5. **Wrong constraint handling**: Penalize infeasibility appropriately